In [1]:
import pandas as pd

Esse notebook utiliza o datatset aria midi unique como ponto de partida do projeto. O arquivo de metadata é analisado para recolher informações sobre os midis disponiveis para uso. Após tratativas, o datatset é separado em dados de treino, teste e validação. Cada dataset possui o nome dos arquivos midis que serão utilizados posteriormente nas etapas seguintes.

In [2]:
pip install scikit-learn

In [3]:
with open("/content/MIDI-LSTM-and-transformer-decoder/metadata.json", "r") as f:
    df = pd.read_json(f)

In [4]:
df_metadata=pd.DataFrame()

for col_name, col_data in df.items():
    raw_values = pd.DataFrame([col_data.values[0]])
    raw_values["scores"] = [col_data.values[1]]
    col_name = str(col_name)
    name = col_name.zfill(6) if len(col_name) <6 else col_name
    raw_values["audio_name"] = name
    df_metadata = pd.concat([raw_values, df_metadata], ignore_index=True)
print(df_metadata)

          composer   opus      genre performer  music_period  \
0            liszt  392.0  classical   hamelin      romantic   
1           ligeti    NaN  classical       NaN  contemporary   
2      saint-saens   56.0  classical       NaN      romantic   
3       moszkowski   68.0  classical       NaN      romantic   
4         respighi   44.0  classical       NaN     classical   
...            ...    ...        ...       ...           ...   
32517     schubert  498.0  classical       NaN      romantic   
32518       franck   10.0  classical       NaN      romantic   
32519      purcell    5.0  classical       NaN       baroque   
32520     maykapar   15.0  classical       NaN           NaN   
32521        weber   77.0  classical       NaN     classical   

                          scores audio_name  piece_number     form  \
0      {'0': 0.9963000000000001}     207047           NaN      NaN   
1      {'0': 0.9904000000000001}     207046           5.0    etude   
2      {'0': 0.992300

In [5]:
df_metadata.columns

Index(['composer', 'opus', 'genre', 'performer', 'music_period', 'scores',
       'audio_name', 'piece_number', 'form', 'key_signature', 'difficulty'],
      dtype='object')

In [6]:
df_metadata.drop(columns=['opus','performer','key_signature','piece_number','form','difficulty'], inplace=True)

In [7]:
df_metadata.columns

Index(['composer', 'genre', 'music_period', 'scores', 'audio_name'], dtype='object')

In [8]:
cleaned_df_metadata = df_metadata.dropna(subset=['genre', 'music_period'])

print(cleaned_df_metadata)

          composer      genre  music_period                     scores  \
0            liszt  classical      romantic  {'0': 0.9963000000000001}   
1           ligeti  classical  contemporary  {'0': 0.9904000000000001}   
2      saint-saens  classical      romantic  {'0': 0.9923000000000001}   
3       moszkowski  classical      romantic  {'0': 0.9902000000000001}   
4         respighi  classical     classical  {'0': 0.9981000000000001}   
...            ...        ...           ...                        ...   
32516      arensky  classical      romantic  {'0': 0.9618000000000001}   
32517     schubert  classical      romantic               {'0': 0.991}   
32518       franck  classical      romantic              {'0': 0.9846}   
32519      purcell  classical       baroque              {'0': 0.9957}   
32521        weber  classical     classical  {'0': 0.9510000000000001}   

      audio_name  
0         207047  
1         207046  
2         207042  
3         207040  
4         207039

In [9]:
print(cleaned_df_metadata.groupby(["genre", "music_period"])["genre"].count())


genre       music_period 
atonal      contemporary        8
            modern             17
blues       contemporary        1
            modern              3
classical   baroque          2291
            classical        7818
            contemporary     1990
            impressionist     854
            modern           1034
            romantic         9817
folk        classical           1
            contemporary        4
            modern              5
            romantic            2
jazz        baroque             1
            classical           2
            contemporary        5
            modern             60
            romantic            1
ragtime     classical           2
            contemporary        4
            modern              2
rock        modern              1
soundtrack  contemporary        8
Name: genre, dtype: int64


In [10]:
print(cleaned_df_metadata.groupby(["genre", "composer"])["genre"].count())


genre       composer  
atonal      bacevicius    1
            berg          3
            ligeti        2
            nancarrow     1
            noland        1
                         ..
ragtime     zerkovitz     1
rock        norton        1
soundtrack  bemani        1
            o'halloran    5
            ohalloran     2
Name: genre, Length: 1888, dtype: int64


In [11]:
most_famous_genre = cleaned_df_metadata[cleaned_df_metadata['genre'] == "classical"].copy()

print(most_famous_genre)

          composer      genre  music_period                     scores  \
0            liszt  classical      romantic  {'0': 0.9963000000000001}   
1           ligeti  classical  contemporary  {'0': 0.9904000000000001}   
2      saint-saens  classical      romantic  {'0': 0.9923000000000001}   
3       moszkowski  classical      romantic  {'0': 0.9902000000000001}   
4         respighi  classical     classical  {'0': 0.9981000000000001}   
...            ...        ...           ...                        ...   
32516      arensky  classical      romantic  {'0': 0.9618000000000001}   
32517     schubert  classical      romantic               {'0': 0.991}   
32518       franck  classical      romantic              {'0': 0.9846}   
32519      purcell  classical       baroque              {'0': 0.9957}   
32521        weber  classical     classical  {'0': 0.9510000000000001}   

      audio_name  
0         207047  
1         207046  
2         207042  
3         207040  
4         207039

In [16]:
from sklearn.model_selection import train_test_split

#Separação dos dados de treinamento, teste e validação
#treinamento = 60%, teste = 25% e validação = 15%
X_train_val, X_test = train_test_split(most_famous_genre, test_size=0.25, random_state=42)

X_train, X_val = train_test_split(X_train_val, test_size=0.15, random_state=42)

In [17]:
X_train


,composer,genre,music_period,scores,audio_name
22110,belkin,classical,contemporary,{'0': 0.9823000000000001},066383
16922,bach,classical,baroque,{'0': 0.9999},098826
31258,orona,classical,contemporary,{'0': 0.9577},007902
30601,scarlatti,classical,baroque,{'0': 0.9947},011944
28156,smith,classical,romantic,{'0': 0.9846},027268
...,...,...,...,...,...
9569,schumann,classical,romantic,{'0': 0.9994000000000001},145538
29460,berlioz,classical,romantic,{'0': 0.9761000000000001},019074
2348,bartok,classical,modern,{'0': 0.9672000000000001},191860
9435,lyadov,classical,romantic,{'0': 0.8183},146460


In [18]:
X_test

,composer,genre,music_period,scores,audio_name
17493,mozart,classical,classical,{'0': 0.9911000000000001},095174
31339,mozart,classical,classical,{'0': 0.9994000000000001},007366
3691,vladigerov,classical,classical,{'0': 0.9886},183335
20678,lemoine,classical,classical,{'0': 0.9994000000000001},075470
2652,prudent,classical,romantic,{'0': 0.9906},189873
...,...,...,...,...,...
5631,mozart,classical,classical,{'0': 0.9874},171427
10790,scarlatti,classical,baroque,{'0': 0.9908},137786
2761,schubert,classical,classical,{'0': 0.929},189232
21383,liapounov,classical,romantic,{'0': 0.9863000000000001},071103


In [19]:
X_val

,composer,genre,music_period,scores,audio_name
467,diabelli,classical,classical,{'0': 0.9837},204085
3171,czerny,classical,romantic,{'0': 0.9845},186635
15089,dussek,classical,classical,{'0': 0.9685},110689
23219,alkan,classical,romantic,{'0': 0.9923000000000001},059307
31942,waldteufel,classical,classical,{'0': 0.9589000000000001},003513
...,...,...,...,...,...
9508,brahms,classical,romantic,{'0': 0.9997},145996
16190,berger,classical,romantic,{'0': 0.9774},103604
20954,scarlatti,classical,baroque,{'0': 0.8200000000000001},073757
27219,weber,classical,romantic,{'0': 0.9262},033167


In [20]:
midi_filenames_train = (X_train["audio_name"].values).astype(str)
len(midi_filenames_train)

15175

In [21]:
midi_filenames_test = (X_test["audio_name"].values).astype(str)
len(midi_filenames_test)

5951

In [22]:
midi_filenames_val = (X_val["audio_name"].values).astype(str)
len(midi_filenames_test)

5951

In [23]:
from pathlib import Path
import shutil
import zipfile

def copy_files(origin, destination, filenames):
  with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as zipf:
    for path_f in origin.rglob('*'):
      if path_f.is_file() and path_f.name.split("_")[0] in filenames:
        zipf.write(path_f, path_f.relative_to(origin))

copy_files(Path("/content/data2"),Path("/content/MIDI-LSTM-and-transformer-decoder/dataset_training.zip"), midi_filenames_train)
copy_files(Path("/content/data2"),Path("/content/MIDI-LSTM-and-transformer-decoder/dataset_test.zip"), midi_filenames_test)
copy_files(Path("/content/data2"),Path("/content/MIDI-LSTM-and-transformer-decoder/dataset_validation.zip"), midi_filenames_val)



